# SafeCircle: stopping scams at the moment money moves

Three people, three payments. For each one SafeCircle reads the message they got,
checks whether the payment fits their normal spending, and decides: let it through,
or pause it and alert a trusted family member. Everything runs on IBM LinuxONE;
private details are removed before any text reaches the language model.

In [20]:
import sys, time, platform
import pandas as pd

REPO = "/home/ibm-user/shared/safecircle"
sys.path.insert(0, REPO)

from src.messages import score_message
from src.explain import redact, explain
from src.transactions import score_transaction
from src.risk import decide

USE_GRANITE = False   # True = IBM Granite writes the explanation (on-server); False = template
print("Running on:", platform.machine())   # s390x = IBM Z / LinuxONE

Running on: s390x


In [21]:
messages     = pd.read_csv(f"{REPO}/scenarios/messages.csv")
transactions = pd.read_csv(f"{REPO}/scenarios/transactions.csv")
messages.columns     = messages.columns.str.strip().str.lower()
transactions.columns = transactions.columns.str.strip().str.lower()
messages["ts"]     = pd.to_datetime(messages["timestamp"])
transactions["ts"] = pd.to_datetime(transactions["timestamp"])
messages = messages.sort_values("ts")

TXN_KEYS = ["user_id", "txn_id", "timestamp", "amount", "merchant",
            "merchant_category", "is_new_payee"]

def find_message(txn_row):
    """The latest message to the same person in the 24 hours before the payment."""
    window = messages[(messages["user_id"] == txn_row["user_id"])
                      & (messages["ts"] <= txn_row["ts"])
                      & (messages["ts"] >= txn_row["ts"] - pd.Timedelta(hours=24))]
    return window.iloc[-1] if len(window) else None

def message_field(txn_row, field):
    m = find_message(txn_row)
    return "" if m is None else str(m[field])

transactions["message_text"]   = transactions.apply(message_field, axis=1, field="text")
transactions["message_sender"] = transactions.apply(message_field, axis=1, field="sender")
transactions["minutes_after"]  = transactions.apply(
    lambda r: None if find_message(r) is None
    else round((r["ts"] - find_message(r)["ts"]).total_seconds() / 60), axis=1)

In [22]:
CATEGORY_WORDS = {"gift_card": "gift-card purchase", "wire": "wire transfer",
                  "money_transfer": "money transfer"}

def family_alert(row):
    name = str(row["user_id"]).capitalize()
    cat = row["merchant_category"]
    what = CATEGORY_WORDS.get(cat, f"{cat.replace('_', ' ')} purchase")
    return (f"SafeCircle paused a ${float(row['amount']):,.0f} {what} on {name}'s card "
            f"after a likely scam text. Please call {name}.")

def run_scenario(user_id):
    row  = transactions[transactions["user_id"] == user_id].iloc[0]
    text = row["message_text"]

    t0 = time.perf_counter()
    msg_result  = score_message(text) if text else {"score": 0.0, "reasons": []}
    txn_result  = score_transaction(row[TXN_KEYS].to_dict())   # contract keys only
    decision    = decide(msg_result, txn_result)
    safe_text   = redact(text)                                   # BEFORE the language model
    explanation = explain(safe_text, decision, use_granite=USE_GRANITE)
    ms = (time.perf_counter() - t0) * 1000

    name = user_id.capitalize()
    held = decision["action"] == "hold_and_alert"
    new_payee = "yes" if int(row["is_new_payee"]) == 1 else "no"

    print(f"================  {name.upper()}  ================")
    print(f"MESSAGE from {redact(row['message_sender'])}:")
    print(f"   \"{text}\"")
    print(f"   What the AI sees (private details hidden): \"{safe_text}\"")
    print(f"\nPAYMENT {row['minutes_after']} min later: ${float(row['amount']):,.2f} at "
          f"{row['merchant']} ({row['merchant_category']}), new payee: {new_payee}")
    print(f"\nSCORES   message {msg_result['score']:.2f} | payment {txn_result['score']:.2f} "
          f"| combined risk {decision['risk']:.2f}")
    print(f"DECISION {'PAUSED + FAMILY ALERTED' if held else 'ALLOWED'}")
    print(f"WHY      {', '.join(decision['reasons']) or 'nothing unusual'}")
    print(f"\nTO {name.upper()}:  {explanation}")
    if held:
        print(f"TO FAMILY: {family_alert(row)}")
    result = "PASS" if decision["action"] == row["expected_action"] else "FAIL"
    print(f"\n[{result}] expected {row['expected_action']}  |  decided in {ms:.1f} ms\n")

    return {"user": name, "expected": row["expected_action"], "decided": decision["action"],
            "risk": decision["risk"], "result": result, "ms": round(ms, 1)}

results = []

## 1. Rosa, 78: the grandparent scam
Rosa gets a text from someone claiming to be her grandchild, asking for gift cards.
Twenty-three minutes later, her card is used for $1,500 of gift cards.

In [23]:
results.append(run_scenario("rosa"))

================  ROSA  ================
MESSAGE from [PHONE]:
   "Grandma it's me, I'm in trouble. Please buy gift cards and text me the codes. Don't tell Mom."
   What the AI sees (private details hidden): "Grandma it's me, I'm in trouble. Please buy gift cards and text me the codes. Don't tell Mom."

PAYMENT 23 min later: $1,500.00 at QuickMart Gift Cards (gift_card), new payee: yes

SCORES   message 0.64 | payment 1.00 | combined risk 0.82
DECISION PAUSED + FAMILY ALERTED
WHY      asks you to text a code, claims to be a family member, 26x usual amount, gift-card or wire payment, new payee, scam-like message just before a gift-card or wire payment

TO ROSA:  We paused this payment because the message asks you to text a code and claims to be a family member. The payment itself also looked unusual: 26x usual amount, gift-card or wire payment and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you alread

## 2. Walter: a normal day
Walter's pharmacy texts that his prescription is ready. He pays $42 at the pharmacy.
SafeCircle should stay out of his way.


In [24]:
results.append(run_scenario("walter"))

================  WALTER  ================
MESSAGE from CVS Pharmacy:
   "Your prescription is ready for pickup."
   What the AI sees (private details hidden): "Your prescription is ready for pickup."

PAYMENT 105 min later: $42.00 at CVS Pharmacy (pharmacy), new payee: no

SCORES   message 0.34 | payment 0.00 | combined risk 0.17
DECISION ALLOWED
WHY      nothing unusual

TO WALTER:  This payment looks normal for you, so it went through. No action is needed.

[PASS] expected allow  |  decided in 8.9 ms



## 3. Linda: the "account locked" scam
Linda gets a fake bank text saying her account is locked. Forty minutes later she
sends a $300 wire to someone she has never paid before.

In [25]:
results.append(run_scenario("linda"))

================  LINDA  ================
MESSAGE from [PHONE]:
   "Your bank account is locked. Verify your identity at the link below."
   What the AI sees (private details hidden): "Your bank account is locked. Verify your identity at the link below."

PAYMENT 40 min later: $300.00 at Wire to J. Smith (wire), new payee: yes

SCORES   message 0.70 | payment 0.79 | combined risk 0.74
DECISION PAUSED + FAMILY ALERTED
WHY      mentions your account, asks you to verify your identity, 3x usual amount, gift-card or wire payment, new payee, scam-like message just before a gift-card or wire payment

TO LINDA:  We paused this payment because the message mentions your account and asks you to verify your identity. The payment itself also looked unusual: 3x usual amount, gift-card or wire payment and new payee. These are common signs of a scam, so we let your trusted contact know. Please call them, or call the person at a number you already know, before sending any money.
TO FAMILY: SafeCircle p

In [26]:
summary = pd.DataFrame(results)
display(summary)
print(f"{(summary['result'] == 'PASS').sum()} / {len(summary)} scenarios PASS")

# Speed: score every scenario 100 times (template explanations) and average
N = 100
t0 = time.perf_counter()
for _ in range(N):
    for _, row in transactions.iterrows():
        m = score_message(row["message_text"])
        t = score_transaction(row[TXN_KEYS].to_dict())
        d = decide(m, t)
        explain(redact(row["message_text"]), d)
per_decision = (time.perf_counter() - t0) * 1000 / (N * len(transactions))
print(f"Average time per payment decision on {platform.machine()}: {per_decision:.2f} ms")

,user,expected,decided,risk,result,ms
0,Rosa,hold_and_alert,hold_and_alert,0.821,PASS,9.5
1,Walter,allow,allow,0.168,PASS,8.9
2,Linda,hold_and_alert,hold_and_alert,0.744,PASS,10.5


3 / 3 scenarios PASS
Average time per payment decision on s390x: 7.66 ms
